# Physics-Informed Reconstruction of Missing Intervals in Martian Radio-Occultation Profiles

**Data:** MRO Radio Science Subsystem temperature–pressure profiles (PDS `MRO-M-RSS-5-TPS-V1.0`, Hinson/SETI)
**Authors:** Shailendra Dabral, Keshav Agrawal

This notebook runs the complete experiment: data audit, leakage-free splits, twelve reconstruction methods,
four experiments, uncertainty calibration, paired significance tests, and all figures and tables
(PNG/PDF figures, CSV and LaTeX tables).

### Structure
| § | Content |
|---|---|
| 0 | Configuration and reproducibility |
| 1 | Data loading, quality control, physics audit |
| 2 | Profile-level splits |
| 3 | Gap scenarios (defined in kilometres) |
| 4 | Physics formulation |
| 5 | Methods: baselines and neural models |
| 6 | Global pre-training |
| 7 | **E1** Main benchmark (7a: hyper-parameter selection on validation profiles) |
| 8 | Metrics, aggregation, statistics · 8b Conformal calibration |
| 9 | **E2** Noise robustness · **E3** Physics-weight sensitivity · **E4** Mars-year hold-out |
| 10 | Figures and tables |
| 11 | Auto-generated results summary, limitations |

## Research questions

- **RQ1 (accuracy).** Does a physics-informed network reconstruct temperature, pressure and number density inside a
  contiguous missing interval more accurately than interpolation, a Gaussian process, a hydrostatic-integration
  baseline and a data-only network with the same architecture and training?
- **RQ2 (where physics helps).** How does the benefit depend on gap size (2–20 km) and gap location
  (lower, middle, upper)? Interior gaps are interpolation; lower and upper gaps are extrapolation.
- **RQ3 (robustness).** How do methods degrade when the observed part of the profile is noisier than the retrieval
  uncertainty? How sensitive is the PINN to the physics-loss weight?
- **RQ4 (uncertainty).** Are deep-ensemble and GP predictive intervals calibrated, and does predicted
  uncertainty rank actual errors?
- **RQ5 (generalisation).** Does the cross-profile prior transfer to an unseen Mars year?

## Scientific framing

The TPS products are retrieved as: bending angle → refractive index (Abel transform) → **number density** →
pressure by **hydrostatic integration** → temperature by the **ideal-gas law**. Both physics constraints
are therefore built into the reference data, with these consequences:

1. Physical consistency of the *reference* is guaranteed, so "physics residual" results show consistency with the
   retrieval physics, not discovery of new physics.
2. The ideal-gas law is enforced **exactly by construction**: the network predicts $(T, \ln P)$ and
   $n = P/(k_B T)$ is derived. There is no ideal-gas loss to tune.
3. Hydrostatic balance is imposed in the **geopotential form** used by the retrieval, with the retrieval's
   mean molecular mass. Physics alone does not determine $T$ inside a gap, so $T$ in the gap comes from the
   cross-profile prior plus the observed data; the physics then couples $\ln P$ (and $n$) to that $T$.
4. The strongest classical competitor is therefore **hydrostatic integration of an interpolated temperature**,
   which is included as a baseline.
5. The reference values carry 1σ uncertainties (σ_T, σ_P). Errors are also reported relative to σ_T.

## 0. Configuration and reproducibility
Set `QUICK_MODE = True` for a fast smoke test (minutes). Set it to `False` for the full experiment.
Results are cached, so an interrupted session resumes where it stopped.

In [ ]:
import os, sys, json, math, time, copy, random, warnings, platform, zlib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from scipy import stats
from scipy.interpolate import PchipInterpolator
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel
from sklearn.exceptions import ConvergenceWarning
import torch
import torch.nn as nn

warnings.filterwarnings("ignore", category=ConvergenceWarning)

CFG = dict(
    # ---- paths ----
    DATA_LEVELS="../data/processed/mro_tps_combined.csv",   # output of mro_tps_loader.py
    OUTPUT_DIR="../results",
    QUICK_MODE=False,

    # ---- data quality control ----
    SIGMA_T_MAX=5.0,        # K. Drop levels where retrieval sigma_T exceeds this (upper-boundary initialisation zone)
    MIN_LEVELS=40,          # drop profiles with fewer levels after trimming

    # ---- splits ----
    SPLIT_FRACS=(0.70, 0.15, 0.15),   # train / val / test, by profile, stratified by Mars year
    SPLIT_SEED=2024,

    # ---- gap scenarios ----
    GAP_SIZES_KM=[2, 5, 10, 20],
    GAP_LOCATIONS=["lower", "middle", "upper"],
    MIN_OBSERVED_KM=5.0,    # a case is used only if at least this much of the profile stays observed
    MAX_TEST_PROFILES=None, # None = all test profiles

    # ---- neural networks ----
    SEEDS=[0, 1, 2, 3, 4],  # deep-ensemble members
    HIDDEN=64, DEPTH=4,
    PRETRAIN_EPOCHS=3000, PRETRAIN_LR=2e-3, EVAL_EVERY=25,
    COLLOC_POOL_PER_PROFILE=200, COLLOC_BATCH=2048,
    FINETUNE_STEPS=400, FINETUNE_LR=5e-4, N_COLLOC_GAP=64,
    LAMBDA_PHYS=1.0,                  # used for pre-training and the plain fine-tuned PINN
    # regularised fine-tuning: L2-SP pull towards pre-trained weights + early stopping
    ES_WINDOW_KM=3.0, ES_EVAL_EVERY=25,   # every 2nd observed level within this distance of the gap is held out
    TUNE_CONFIGS=[(10, "middle"), (20, "lower"), (20, "upper")],   # validation cases used for selection
    TUNE_ALPHAS=[0.0, 0.1, 1.0], TUNE_LAMBDAS=[0.1, 1.0], TUNE_ES=[True, False],
    # conformal calibration on validation profiles
    CAL_GAP_SIZES_KM=[2, 5, 10, 20],
    # EOF (DINEOF-style) baseline
    EOF_DZ_KM=0.5, EOF_VAR_FRAC=0.99, EOF_MAX_K=15, EOF_MIN_COVER=0.2,
    SIGMA_WEIGHTING=True, SIGMA_FLOOR_T=0.5, SIGMA_FLOOR_LNP=0.003,

    # ---- extra experiments ----
    RUN_NOISE=True, NOISE_MULTIPLIERS=[0, 2, 5], NOISE_CASES=[(10, "middle")], NOISE_SEEDS=3,
    RUN_LAMBDA_SWEEP=True, LAMBDAS=[0.0, 0.01, 0.1, 1.0, 10.0, 100.0], LAMBDA_CASES=[(10, "middle"), (10, "upper")],
    RUN_YEAR_HOLDOUT=True, HOLDOUT_CASES=[(10, "middle"), (10, "upper")], HOLDOUT_SEEDS=3, HOLDOUT_MAX_TEST=20,

    # ---- statistics ----
    N_BOOT=5000, ALPHA=0.05,
)

if CFG["QUICK_MODE"]:
    CFG.update(GAP_SIZES_KM=[5, 10], GAP_LOCATIONS=["middle", "upper"], SEEDS=[0, 1], MAX_TEST_PROFILES=5,
               PRETRAIN_EPOCHS=300, FINETUNE_STEPS=100, NOISE_SEEDS=2, HOLDOUT_SEEDS=2, HOLDOUT_MAX_TEST=4,
               LAMBDAS=[0.0, 1.0, 10.0], N_BOOT=500, TUNE_ALPHAS=[0.0, 1.0], TUNE_LAMBDAS=[1.0], TUNE_ES=[True, False],
               TUNE_CONFIGS=[(10, "middle"), (10, "upper")], CAL_GAP_SIZES_KM=[5, 10])

OUT = Path(CFG["OUTPUT_DIR"])
FIG, TAB, CKPT, RES = OUT / "figures", OUT / "tables", OUT / "checkpoints", OUT / "raw"
for d in (FIG, TAB, CKPT, RES):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(0)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10, "axes.grid": True,
                     "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})

def savefig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{name}.{ext}", bbox_inches="tight")

env = dict(python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__, torch=torch.__version__,
           platform=platform.platform(), device=str(DEVICE))
json.dump(dict(config=CFG, environment=env), open(OUT / "run_config.json", "w"), indent=2, default=str)
print(json.dumps(env, indent=2)); print("QUICK_MODE =", CFG["QUICK_MODE"])
T_START = time.time()

## 1. Data loading, quality control and physics audit

### Coordinates
The vertical coordinate is the **geopotential height** $Z = \Phi / g_0$, where $\Phi$ is the geopotential
above the reference geoid given in every TPS row (computed by the retrieval from the full gravity model `jgmro_110c`)
and $g_0 = 3.71\ \mathrm{m\,s^{-2}}$ is a conventional constant. Hydrostatic balance is exact in this coordinate:

$$\frac{d\ln P}{d\Phi} = -\frac{m}{k_B T}\quad\Longrightarrow\quad \frac{d\ln P}{dZ} = -\frac{m g_0}{k_B T} = -\frac{1}{H(T)} .$$

This avoids the ≈1.5 % error of using $g=\mu/r^2$ (which ignores rotation and the non-spherical gravity field).
$\Phi$ (and latitude/longitude) at each level follow from the observing geometry, so they are known inside a gap;
using them is not leakage.

### Quality control
Each profile is retrieved downward from an assumed temperature at the top, so σ_T rises to ≈10 K there.
Levels with σ_T > `SIGMA_T_MAX` are removed so that upper-gap results are not dominated by that assumption.

In [ ]:
K_B, N_A = 1.380649e-23, 6.02214076e23
G0 = 3.71          # m s^-2; conventional constant defining geopotential height (choice does not affect physics)
T_REF = 200.0      # K; reference temperature for nondimensionalisation

lv = pd.read_csv(CFG["DATA_LEVELS"])
REQUIRED = ["Profile_ID", "Mars_Year", "Solar_Long", "Radius", "Latitude", "Longitude", "Geopotential",
            "Pressure", "Sigma_P", "Temperature", "Sigma_T", "Number_Density", "Sigma_n"]
missing = [c for c in REQUIRED if c not in lv.columns]
assert not missing, f"missing columns: {missing}"

lv = lv.sort_values(["Profile_ID", "Radius"]).reset_index(drop=True)
n_prof_raw, n_lev_raw = lv["Profile_ID"].nunique(), len(lv)

lv["Z_km"] = lv["Geopotential"] / G0 / 1e3
lv["lnP"] = np.log(lv["Pressure"])
lv["sig_lnP"] = lv["Sigma_P"] / lv["Pressure"]
lon, ls = np.radians(lv["Longitude"]), np.radians(lv["Solar_Long"])
lv["sin_lon"], lv["cos_lon"] = np.sin(lon), np.cos(lon)
lv["sin_ls"], lv["cos_ls"] = np.sin(ls), np.cos(ls)

# --- QC: trim top initialisation zone, require monotonic geopotential, minimum length ---
z_top_before = lv.groupby("Profile_ID")["Z_km"].max()
lv = lv[lv["Sigma_T"] <= CFG["SIGMA_T_MAX"]].copy()
z_top_after = lv.groupby("Profile_ID")["Z_km"].max()
trimmed_km = (z_top_before - z_top_after.reindex(z_top_before.index)).dropna()

mono = pd.Series({pid: bool(np.all(np.diff(z.values) > 0)) for pid, z in lv.groupby("Profile_ID")["Z_km"]})
counts = lv.groupby("Profile_ID").size()
keep = mono[mono].index.intersection(counts[counts >= CFG["MIN_LEVELS"]].index)
lv = lv[lv["Profile_ID"].isin(keep)].reset_index(drop=True)

meta = lv.groupby("Profile_ID").agg(Mars_Year=("Mars_Year", "first"), Solar_Long=("Solar_Long", "first"),
                                    lat=("Latitude", "mean"), lon=("Longitude", "mean"),
                                    n_levels=("Z_km", "size"), z_min=("Z_km", "min"), z_max=("Z_km", "max"))
if "LTST_hr" in lv.columns:
    meta["LTST_hr"] = lv.groupby("Profile_ID")["LTST_hr"].first()
meta["span_km"] = meta["z_max"] - meta["z_min"]

qc = dict(profiles_raw=n_prof_raw, levels_raw=n_lev_raw, profiles_kept=len(meta), levels_kept=len(lv),
          median_top_trim_km=float(trimmed_km.median()), median_span_km=float(meta["span_km"].median()),
          median_levels=float(meta["n_levels"].median()))
print(json.dumps(qc, indent=2))
print("\nProfiles per Mars year and Ls range (checks year/season confounding):")
print(meta.groupby("Mars_Year")["Solar_Long"].agg(["count", "min", "max"]).round(1))

In [ ]:
# --- Physics audit of the reference data ---
def implied_mass(g):
    T, lnP, Phi = g["Temperature"].values, g["lnP"].values, g["Geopotential"].values
    dPhi = np.diff(Phi); ok = dPhi > 0
    m = K_B * 0.5 * (T[1:] + T[:-1])[ok] * (-np.diff(lnP))[ok] / dPhi[ok]
    return np.median(m)

m_prof = pd.Series({pid: implied_mass(g) for pid, g in lv.groupby("Profile_ID")})
M_MEAN = float(np.median(m_prof))                       # kg per molecule (retrieval constant)
H_REF_KM = K_B * T_REF / (M_MEAN * G0) / 1e3            # reference scale height, km

ideal_dev = np.abs(lv["Pressure"] / (lv["Number_Density"] * K_B * lv["Temperature"]) - 1)

def hydro_residual_np(z_km, T, lnP):
    """Dimensionless hydrostatic residual  H_ref dlnP/dZ + T_ref/T  (0 when exact)."""
    o = np.argsort(z_km)
    return H_REF_KM * np.gradient(lnP[o], z_km[o]) + T_REF / T[o]

ref_res = pd.Series({pid: np.sqrt(np.mean(hydro_residual_np(g["Z_km"].values, g["Temperature"].values,
                                                           g["lnP"].values) ** 2)) for pid, g in lv.groupby("Profile_ID")})
REF_HYDRO_FLOOR = float(np.median(ref_res))

audit = dict(mean_molecular_mass_gmol=M_MEAN * N_A * 1e3,
             molecular_mass_IQR_gmol=[float(q * N_A * 1e3) for q in m_prof.quantile([0.25, 0.75])],
             H_ref_km=H_REF_KM, ideal_gas_max_rel_dev=float(ideal_dev.max()),
             reference_hydro_residual_RMS_median=REF_HYDRO_FLOOR, **qc)
json.dump(audit, open(OUT / "data_audit.json", "w"), indent=2)
print(json.dumps(audit, indent=2))
print("\nThe reference hydrostatic residual is the floor set by finite differencing and the rounding of the")
print("geopotential column; no method can be meaningfully 'more hydrostatic' than this.")

In [ ]:
# --- Figure 1: data overview ---
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
years = sorted(meta["Mars_Year"].unique())
cmap = plt.get_cmap("viridis", len(years))
for i, my in enumerate(years):
    s = meta[meta["Mars_Year"] == my]
    ax[0].scatter(s["Solar_Long"], s["lat"], s=18, color=cmap(i), label=f"MY {int(my)}")
ax[0].set_xlabel("Solar longitude $L_s$ (deg)"); ax[0].set_ylabel("Latitude (deg)")
ax[0].set_title("(a) Coverage"); ax[0].legend(fontsize=8)
for pid, g in lv.groupby("Profile_ID"):
    i = years.index(meta.loc[pid, "Mars_Year"])
    ax[1].plot(g["Temperature"], g["Z_km"], lw=0.5, alpha=0.5, color=cmap(i))
ax[1].set_xlabel("Temperature (K)"); ax[1].set_ylabel("Geopotential height Z (km)"); ax[1].set_title("(b) All profiles")
lv["_h_top"] = lv.groupby("Profile_ID")["Z_km"].transform("max") - lv["Z_km"]
ax[2].scatter(lv["Sigma_T"], lv["_h_top"], s=2, alpha=0.3)
ax[2].set_xlabel(r"Retrieval $\sigma_T$ (K)"); ax[2].set_ylabel("Distance below (trimmed) top (km)")
ax[2].invert_yaxis(); ax[2].set_title(r"(c) Retrieval uncertainty")
lv.drop(columns="_h_top", inplace=True)
fig.tight_layout(); savefig(fig, "fig01_data_overview"); plt.show()

## 2. Profile-level splits
Whole profiles are assigned to train / validation / test (stratified by Mars year), so no level of a test profile
is ever seen during pre-training. The secondary **Mars-year hold-out** split (E4) is built in §9.

In [ ]:
def stratified_profile_split(meta, fracs, seed):
    rng = np.random.default_rng(seed)
    tr, va, te = [], [], []
    for _, g in meta.groupby("Mars_Year"):
        ids = list(rng.permutation(g.index.values)); n = len(ids)
        if n < 3:
            tr += ids; continue
        n_te = max(1, int(round(fracs[2] * n))); n_va = max(1, int(round(fracs[1] * n)))
        te += ids[:n_te]; va += ids[n_te:n_te + n_va]; tr += ids[n_te + n_va:]
    return sorted(tr), sorted(va), sorted(te)

TRAIN_IDS, VAL_IDS, TEST_IDS = stratified_profile_split(meta, CFG["SPLIT_FRACS"], CFG["SPLIT_SEED"])
assert not (set(TRAIN_IDS) & set(VAL_IDS) or set(TRAIN_IDS) & set(TEST_IDS) or set(VAL_IDS) & set(TEST_IDS))
if CFG["MAX_TEST_PROFILES"]:
    TEST_IDS = TEST_IDS[:CFG["MAX_TEST_PROFILES"]]

json.dump(dict(train=TRAIN_IDS, val=VAL_IDS, test=TEST_IDS), open(OUT / "splits.json", "w"), indent=2)
train_df = lv[lv["Profile_ID"].isin(TRAIN_IDS)].reset_index(drop=True)
val_df = lv[lv["Profile_ID"].isin(VAL_IDS)].reset_index(drop=True)
test_df = lv[lv["Profile_ID"].isin(TEST_IDS)].reset_index(drop=True)
for name, ids in [("train", TRAIN_IDS), ("val", VAL_IDS), ("test", TEST_IDS)]:
    print(f"{name:5s}: {len(ids):3d} profiles | per Mars year:",
          meta.loc[ids].groupby("Mars_Year").size().to_dict())

## 3. Gap scenarios
Gaps are defined in **kilometres of geopotential height** (2, 5, 10, 20 km), because the number of levels per km
differs by more than a factor of two between profiles. Locations: **lower** (from the base; extrapolation downward),
**middle** (centred; interpolation), **upper** (to the top; extrapolation upward). Inside a gap, no $T$, $P$ or $n$
value is used by any method. A case is kept only if ≥ `MIN_OBSERVED_KM` of the profile remains observed.

In [ ]:
def make_gap(z, size_km, location, min_obs_km=CFG["MIN_OBSERVED_KM"]):
    z0, z1 = z.min(), z.max()
    if (z1 - z0) - size_km < min_obs_km:
        return None
    if location == "lower":
        lo, hi = z0 - 1e-9, z0 + size_km
    elif location == "upper":
        lo, hi = z1 - size_km, z1 + 1e-9
    else:
        c = 0.5 * (z0 + z1); lo, hi = c - size_km / 2, c + size_km / 2
    gap = (z >= lo) & (z <= hi)
    if gap.sum() < 3 or (~gap).sum() < 5:
        return None
    return gap

CASES = []
for pid in TEST_IDS:
    z = test_df.loc[test_df["Profile_ID"] == pid, "Z_km"].values
    for s in CFG["GAP_SIZES_KM"]:
        for loc in CFG["GAP_LOCATIONS"]:
            if make_gap(z, s, loc) is not None:
                CASES.append((pid, s, loc))
n_possible = len(TEST_IDS) * len(CFG["GAP_SIZES_KM"]) * len(CFG["GAP_LOCATIONS"])
print(f"Main benchmark cases: {len(CASES)} of {n_possible} possible (others too short for the gap)")

## 4. Physics formulation

**Outputs.** The network maps $x=(Z,\ \text{lat},\ \sin\lambda,\cos\lambda,\ \sin L_s,\cos L_s)$ to $(T,\ \ln P)$.
Number density is derived, $n = P/(k_B T)$, so the ideal-gas law holds exactly.

**Hydrostatic residual (dimensionless).** With $H_{\rm ref} = k_B T_{\rm ref}/(m g_0)$,

$$ r_{\rm hyd}(x) = H_{\rm ref}\,\frac{\partial \ln \hat P}{\partial Z} + \frac{T_{\rm ref}}{\hat T} ,$$

which is zero for exact hydrostatic balance and $O(1)$ in magnitude, so no ad-hoc scale factor is needed.
The network normalises inputs and de-normalises outputs internally, so autograd returns
$\partial \ln \hat P / \partial Z$ directly in physical units (no hand-written chain-rule factor).

**Loss.**
$$\mathcal L = \frac{1}{N}\sum_i \sum_{k\in\{T,\ln P\}} w_{ik}\Big(\frac{\hat y_{ik}-y_{ik}}{s_k}\Big)^2
 + \lambda\,\frac{1}{N_c}\sum_j r_{\rm hyd}(x_j)^2 ,\qquad
w_{ik}\propto \frac{1}{\max(\sigma_{ik},\sigma_{k,\rm floor})^2}$$

where $s_k$ is the training-set standard deviation and $w$ uses the retrieval's own 1σ uncertainties
(normalised to mean 1 over the training set). Collocation points $x_j$ are sampled densely in $Z$ along each profile's
geometry, including inside gaps (coordinates only).

In [ ]:
IN_COLS = ["Z_km", "Latitude", "sin_lon", "cos_lon", "sin_ls", "cos_ls"]
OUT_COLS = ["Temperature", "lnP"]

def compute_norm(df):
    return dict(in_mu=df[IN_COLS].mean().values, in_sd=np.maximum(df[IN_COLS].std().values, 1e-6),
                out_mu=df[OUT_COLS].mean().values, out_sd=df[OUT_COLS].std().values)

def sigma_eff(df):
    return np.column_stack([np.maximum(df["Sigma_T"].values, CFG["SIGMA_FLOOR_T"]),
                            np.maximum(df["sig_lnP"].values, CFG["SIGMA_FLOOR_LNP"])])

NORM = compute_norm(train_df)
W_NORM = (1.0 / sigma_eff(train_df) ** 2).mean(axis=0)

def to_t(a):
    return torch.as_tensor(np.asarray(a, dtype=np.float32), device=DEVICE)

def xyw(df):
    W = (1.0 / sigma_eff(df) ** 2) / W_NORM if CFG["SIGMA_WEIGHTING"] else np.ones((len(df), 2))
    return to_t(df[IN_COLS].values), to_t(df[OUT_COLS].values), to_t(W)

class ProfileNet(nn.Module):
    """Tanh MLP with internal input/output scaling; forward() works in physical units."""
    def __init__(self, norm, hidden=CFG["HIDDEN"], depth=CFG["DEPTH"]):
        super().__init__()
        for k, v in norm.items():
            self.register_buffer(k, torch.as_tensor(np.asarray(v, dtype=np.float32)))
        layers, d = [], len(IN_COLS)
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.Tanh()]; d = hidden
        layers.append(nn.Linear(d, len(OUT_COLS)))
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net((x - self.in_mu) / self.in_sd) * self.out_sd + self.out_mu

def data_loss(model, X, Y, W):
    r = (model(X) - Y) / model.out_sd
    return (W * r ** 2).mean()

def hydro_residual_torch(model, Xc):
    Xc = Xc.detach().clone().requires_grad_(True)
    Y = model(Xc)
    dlnP = torch.autograd.grad(Y[:, 1].sum(), Xc, create_graph=True)[0][:, 0]   # d lnP / dZ_km
    return H_REF_KM * dlnP + T_REF / torch.clamp(Y[:, 0], min=50.0)

def densify(prof, n, rng, z_lo=None, z_hi=None):
    """Collocation coordinates along a profile's geometry (no atmospheric values used)."""
    z = prof["Z_km"].values
    zs = rng.uniform(z.min() if z_lo is None else z_lo, z.max() if z_hi is None else z_hi, n)
    lat = np.interp(zs, z, prof["Latitude"].values)
    sl, cl = np.interp(zs, z, prof["sin_lon"].values), np.interp(zs, z, prof["cos_lon"].values)
    r = np.hypot(sl, cl)
    return np.column_stack([zs, lat, sl / r, cl / r,
                            np.full(n, prof["sin_ls"].iloc[0]), np.full(n, prof["cos_ls"].iloc[0])])

@torch.no_grad()
def predict(model, df):
    Y = model(to_t(df[IN_COLS].values)).cpu().numpy()
    return Y[:, 0], Y[:, 1]

print(f"H_ref = {H_REF_KM:.2f} km, m = {M_MEAN * N_A * 1e3:.3f} g/mol, weights normalised by {W_NORM}")

## 5. Methods

| Method | Uses other profiles | Uses physics | Uncertainty |
|---|---|---|---|
| Linear (in $T$, $\ln P$) | no | no | no |
| PCHIP (in $T$, $\ln P$) | no | no | no |
| **Hydrostatic integration** (PCHIP $T$, integrate $d\ln P/dZ=-1/H(T)$ from the gap edges, blended) | no | yes (exact) | no |
| Gaussian process (RBF + white noise, linear mean) | no | no | yes |
| **EOF / optimal interpolation** (DINEOF-filled EOFs of training profiles; Bayesian conditioning on observed levels) | yes | no | yes |
| Random forest (climatology, zero-shot) | yes | no | no |
| MLP / PINN zero-shot (pre-trained only) | yes | PINN: yes | ensemble |
| **MLP fine-tuned** (data-only) | yes | no | ensemble |
| PINN fine-tuned (plain; λ = 1, fixed steps) | yes | yes | ensemble |
| MLP fine-tuned, regularised | yes | no | ensemble |
| **PINN fine-tuned, regularised (ours)** | yes | yes | ensemble |

**Regularised fine-tuning.** Plain fine-tuning can drift away from the pre-trained climatology inside large
extrapolation gaps. The regularised version adds an L2-SP penalty $\alpha\sum_k(\theta_k-\theta_k^{(0)})^2$ towards the
pre-trained weights and early stopping on held-out observed levels: every second observed level within
`ES_WINDOW_KM` of the gap is withheld and the checkpoint with the lowest held-out data loss is kept. Step 0 (no
fine-tuning) is a candidate, so the model can fall back to its prior. Early stopping judges only the data fit next
to the gap, so whether it is used is itself selected on validation profiles. $\alpha$ and $\lambda$ are chosen on
**validation profiles only** (§7a).

MLP and PINN use **identical** architecture, pre-training schedule, fine-tuning schedule, seeds and model-selection
rule; they differ only in $\lambda$. Extrapolation outside the observed range (lower/upper gaps) for the
interpolators uses a linear fit to the nearest 3 km of observed data.

In [ ]:
def interp_extrap(z_obs, y_obs, z_new, kind="linear", window_km=3.0):
    o = np.argsort(z_obs); z_obs, y_obs = z_obs[o], y_obs[o]
    out = np.empty(len(z_new))
    inside = (z_new >= z_obs[0]) & (z_new <= z_obs[-1])
    if kind == "pchip":
        out[inside] = PchipInterpolator(z_obs, y_obs)(z_new[inside])
    else:
        out[inside] = np.interp(z_new[inside], z_obs, y_obs)
    for side in ("below", "above"):
        sel = z_new < z_obs[0] if side == "below" else z_new > z_obs[-1]
        if sel.any():
            near = z_obs <= z_obs[0] + window_km if side == "below" else z_obs >= z_obs[-1] - window_km
            if near.sum() < 2:
                near = np.zeros_like(near); near[:2] = side == "below"; near[-2:] = side == "above"
            c = np.polyfit(z_obs[near], y_obs[near], 1)
            out[sel] = np.polyval(c, z_new[sel])
    return out

def hydrostatic_integration(z_obs, T_obs, lnP_obs, z_gap):
    T_gap = interp_extrap(z_obs, T_obs, z_gap, "pchip")
    z_all = np.concatenate([z_obs, z_gap]); T_all = np.concatenate([T_obs, T_gap])
    o = np.argsort(z_all); z_all, T_all = z_all[o], T_all[o]
    zf = np.linspace(z_all[0], z_all[-1], max(2000, 20 * len(z_all)))
    Tf = np.interp(zf, z_all, T_all)
    integrand = -T_REF / (H_REF_KM * Tf)                             # d lnP / dZ
    I = np.concatenate([[0.0], np.cumsum(0.5 * (integrand[1:] + integrand[:-1]) * np.diff(zf))])
    Ig = np.interp(z_gap, zf, I)
    below, above = z_obs < z_gap.min(), z_obs > z_gap.max()
    preds, wts = [], []
    if below.any():
        zb = z_obs[below].max(); lb = lnP_obs[below][np.argmax(z_obs[below])]
        preds.append(lb + Ig - np.interp(zb, zf, I))
    if above.any():
        za = z_obs[above].min(); la = lnP_obs[above][np.argmin(z_obs[above])]
        preds.append(la + Ig - np.interp(za, zf, I))
    if len(preds) == 2:
        w = (z_gap - zb) / (za - zb)
        lnP_gap = (1 - w) * preds[0] + w * preds[1]
    else:
        lnP_gap = preds[0]
    return T_gap, lnP_gap

def gp_predict(z_obs, y_obs, z_new):
    c = np.polyfit(z_obs, y_obs, 1)
    kern = ConstantKernel(1.0, (1e-3, 1e3)) * RBF(5.0, (0.3, 50.0)) + WhiteKernel(1e-2, (1e-8, 1.0))
    gp = GaussianProcessRegressor(kern, normalize_y=True, n_restarts_optimizer=2, random_state=0)
    gp.fit(z_obs[:, None], y_obs - np.polyval(c, z_obs))
    mu, sd = gp.predict(z_new[:, None], return_std=True)
    return mu + np.polyval(c, z_new), sd

def fit_rf(df, seed=0):
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=3, n_jobs=-1, random_state=seed)
    rf.fit(df[IN_COLS].values, df[OUT_COLS].values)
    return rf

def fit_eof(df, var, sig_col):
    """DINEOF-style EOFs on a common geopotential-height grid (gaps filled iteratively by low-rank SVD)."""
    dz = CFG["EOF_DZ_KM"]
    grid = np.arange(np.floor(df["Z_km"].min()), np.ceil(df["Z_km"].max()) + dz / 2, dz)
    groups = list(df.groupby("Profile_ID"))
    X = np.full((len(groups), len(grid)), np.nan)
    for i, (_, g) in enumerate(groups):
        z, y = g["Z_km"].values, g[var].values
        inside = (grid >= z.min()) & (grid <= z.max())
        X[i, inside] = np.interp(grid[inside], z, y)
    keep = np.isfinite(X).mean(0) >= CFG["EOF_MIN_COVER"]
    grid, X = grid[keep], X[:, keep]
    mask = np.isfinite(X)
    Xf = np.where(mask, X, np.nanmean(X, 0))
    k = None
    for it in range(200):
        mu = Xf.mean(0); U, S, Vt = np.linalg.svd(Xf - mu, full_matrices=False)
        if k is None:
            frac = np.cumsum(S ** 2) / np.sum(S ** 2)
            k = int(min(CFG["EOF_MAX_K"], np.searchsorted(frac, CFG["EOF_VAR_FRAC"]) + 1))
        R = (U[:, :k] * S[:k]) @ Vt[:k] + mu
        delta = np.max(np.abs(Xf[~mask] - R[~mask])) if (~mask).any() else 0.0
        Xf[~mask] = R[~mask]
        if delta < 1e-6 * (np.abs(mu).mean() + 1e-12):
            break
    mu = Xf.mean(0); U, S, Vt = np.linalg.svd(Xf - mu, full_matrices=False)
    R = (U[:, :k] * S[:k]) @ Vt[:k] + mu
    lam_ = S[:k] ** 2 / max(len(X) - 1, 1)
    sig2 = float(np.mean((X[mask] - R[mask]) ** 2))
    return dict(grid=grid, mu=mu, V=Vt[:k].T, lam=lam_, sig2=sig2, k=k, var=var, sig_col=sig_col)

def eof_predict(E, z_obs, y_obs, sig_obs, z_new):
    g = E["grid"]
    zo, zn = np.clip(z_obs, g[0], g[-1]), np.clip(z_new, g[0], g[-1])
    Uo = np.column_stack([np.interp(zo, g, E["V"][:, j]) for j in range(E["k"])])
    Un = np.column_stack([np.interp(zn, g, E["V"][:, j]) for j in range(E["k"])])
    mo, mn = np.interp(zo, g, E["mu"]), np.interp(zn, g, E["mu"])
    rinv = 1.0 / (E["sig2"] + sig_obs ** 2)
    Sa = np.linalg.inv(np.diag(1.0 / E["lam"]) + Uo.T @ (rinv[:, None] * Uo))
    ma = Sa @ (Uo.T @ (rinv * (y_obs - mo)))
    var = np.einsum("ij,jk,ik->i", Un, Sa, Un) + E["sig2"]
    return mn + Un @ ma, np.sqrt(var)

def fit_eof_pair(df):
    return dict(T=fit_eof(df, "Temperature", "Sigma_T"), P=fit_eof(df, "lnP", "sig_lnP"))

def finetune_reg(base, obs_df, colloc, lam, alpha, gap_lo, gap_hi, use_es=True,
                 steps=CFG["FINETUNE_STEPS"], lr=CFG["FINETUNE_LR"]):
    """Fine-tuning with L2-SP towards the pre-trained weights and early stopping on gap-adjacent observed levels."""
    obs_df = obs_df.sort_values("Z_km").reset_index(drop=True)
    z, W_ = obs_df["Z_km"].values, CFG["ES_WINDOW_KM"]
    hold = np.zeros(len(z), bool)
    for side in (np.where((z < gap_lo) & (z >= gap_lo - W_))[0][::-1], np.where((z > gap_hi) & (z <= gap_hi + W_))[0]):
        hold[side[::2]] = True        # every 2nd level, starting next to the gap
    if not use_es or hold.sum() < 2 or (~hold).sum() < 5:
        hold[:] = False
    model = copy.deepcopy(base); model.train()
    theta0 = [p.detach().clone() for p in model.parameters()]
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X, Y, W = xyw(obs_df[~hold]); Xc = to_t(colloc)
    es = hold.any()
    if es:
        Xe, Ye, We = xyw(obs_df[hold])
        with torch.no_grad():
            best = data_loss(model, Xe, Ye, We).item()
        best_state, best_step = copy.deepcopy(model.state_dict()), 0
    for step in range(1, steps + 1):
        opt.zero_grad()
        loss = data_loss(model, X, Y, W)
        if lam > 0:
            loss = loss + lam * (hydro_residual_torch(model, Xc) ** 2).mean()
        if alpha > 0:
            loss = loss + alpha * sum(((p - p0) ** 2).sum() for p, p0 in zip(model.parameters(), theta0))
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if es and step % CFG["ES_EVAL_EVERY"] == 0:
            with torch.no_grad():
                v = data_loss(model, Xe, Ye, We).item()
            if v < best:
                best, best_state, best_step = v, copy.deepcopy(model.state_dict()), step
    if es:
        model.load_state_dict(best_state)
    model.eval()
    return model

def finetune(base, obs_df, colloc, lam, steps=CFG["FINETUNE_STEPS"], lr=CFG["FINETUNE_LR"]):
    model = copy.deepcopy(base); model.train()
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    X, Y, W = xyw(obs_df); Xc = to_t(colloc)
    for _ in range(steps):
        opt.zero_grad()
        loss = data_loss(model, X, Y, W)
        if lam > 0:
            loss = loss + lam * (hydro_residual_torch(model, Xc) ** 2).mean()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
    model.eval()
    return model

## 6. Global pre-training
One network per (family, seed) is trained on all training profiles (full profiles, no gaps), with checkpoint
selection by **validation data loss** — a rule fixed in advance and identical for both families.
Each test case then fine-tunes a copy on that profile's observed levels only. This is ≈20× cheaper than training
from scratch per case and is how the method would be used operationally.

In [ ]:
LAM = {"MLP": 0.0, "PINN": CFG["LAMBDA_PHYS"]}

def colloc_pool(df, per_profile, rng):
    return np.vstack([densify(g, per_profile, rng) for _, g in df.groupby("Profile_ID")])

def pretrain(tr_df, va_df, family, lam, seed, tag="main", norm=None):
    ckpt = CKPT / f"pretrain_{tag}_{family}_seed{seed}.pt"
    set_seed(seed)                       # BEFORE weight initialisation, so ensemble members differ
    model = ProfileNet(norm if norm is not None else NORM).to(DEVICE)
    if ckpt.exists():
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); model.eval()
        return model
    rng = np.random.default_rng(seed)
    X, Y, W = xyw(tr_df); Xv, Yv, Wv = xyw(va_df)
    pool = to_t(colloc_pool(tr_df, CFG["COLLOC_POOL_PER_PROFILE"], rng))
    E = CFG["PRETRAIN_EPOCHS"]
    opt = torch.optim.Adam(model.parameters(), lr=CFG["PRETRAIN_LR"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=E, eta_min=1e-5)
    best, best_state, hist = np.inf, None, []
    for ep in range(1, E + 1):
        model.train(); opt.zero_grad()
        ld = data_loss(model, X, Y, W); loss = ld; lp = torch.tensor(0.0)
        if lam > 0:
            idx = torch.randint(len(pool), (min(CFG["COLLOC_BATCH"], len(pool)),), device=DEVICE)
            lp = (hydro_residual_torch(model, pool[idx]) ** 2).mean(); loss = loss + lam * lp
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sched.step()
        if ep % CFG["EVAL_EVERY"] == 0 or ep == E:
            model.eval()
            with torch.no_grad():
                vl = data_loss(model, Xv, Yv, Wv).item()
            hist.append(dict(epoch=ep, train_data=ld.item(), train_phys=lp.item(), val_data=vl))
            if vl < best:
                best, best_state = vl, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state); model.eval()
    torch.save(best_state, ckpt)
    pd.DataFrame(hist).to_csv(RES / f"history_{tag}_{family}_seed{seed}.csv", index=False)
    return model

t0 = time.time()
BASE = {fam: [] for fam in LAM}
for fam, lam in LAM.items():
    for s in CFG["SEEDS"]:
        BASE[fam].append((s, pretrain(train_df, val_df, fam, lam, s)))
        print(f"pre-trained {fam} seed {s}  ({time.time() - t0:.0f}s elapsed)")
RF_MAIN = fit_rf(train_df)
EOF_MAIN = fit_eof_pair(train_df)
print(f"EOF modes kept: T {EOF_MAIN['T']['k']}, lnP {EOF_MAIN['P']['k']}")
print(f"Pre-training done in {time.time() - t0:.0f}s")

## 7. E1 — Main benchmark
For every (test profile, gap size, location) all methods predict the hidden levels. Pointwise predictions are saved,
and all metrics are computed afterwards from that single table. The loop caches progress to disk, so a
restarted session continues where it stopped.

In [ ]:
NAMES = dict(lin="Linear", pchip="PCHIP", hydro="Hydrostatic int.", gp="Gaussian process", eof="EOF (opt. interp.)",
             rf="Random forest", mlp_zs="MLP (zero-shot)", pinn_zs="PINN (zero-shot)",
             mlp_ft="MLP (fine-tuned)", pinn_ft="PINN (fine-tuned)",
             mlp_rft="MLP (reg. fine-tuned)", pinn_rft="PINN (reg. fine-tuned)")
OURS = NAMES["pinn_rft"]
ALL_METHODS = list(NAMES)

def case_seed(case):
    return zlib.crc32("|".join(map(str, case)).encode()) % 2**31   # reproducible across sessions

def run_case(prof, gap, base, rf, methods=ALL_METHODS, lam_override=None, noise_mult=0.0,
             rng_seed=0, seeds_limit=None, eof=None, hp=None, hp_override=None):
    """Return pointwise predictions for one profile with one gap."""
    rng = np.random.default_rng(rng_seed)
    obs, hid = prof[~gap].copy(), prof[gap]
    if noise_mult > 0:
        se = sigma_eff(obs)
        obs["Temperature"] += rng.normal(0, noise_mult * se[:, 0])
        obs["lnP"] += rng.normal(0, noise_mult * se[:, 1])
    zo, To, Po, zh = obs["Z_km"].values, obs["Temperature"].values, obs["lnP"].values, hid["Z_km"].values
    P = {}
    if "lin" in methods:
        P["lin"] = (interp_extrap(zo, To, zh), interp_extrap(zo, Po, zh), None, None)
    if "pchip" in methods:
        P["pchip"] = (interp_extrap(zo, To, zh, "pchip"), interp_extrap(zo, Po, zh, "pchip"), None, None)
    if "hydro" in methods:
        P["hydro"] = (*hydrostatic_integration(zo, To, Po, zh), None, None)
    if "gp" in methods:
        (mT, sT), (mP, sP) = gp_predict(zo, To, zh), gp_predict(zo, Po, zh)
        P["gp"] = (mT, mP, sT, sP)
    if "rf" in methods and rf is not None:
        y = rf.predict(hid[IN_COLS].values); P["rf"] = (y[:, 0], y[:, 1], None, None)
    if "eof" in methods and eof is not None:
        mT, sT = eof_predict(eof["T"], zo, To, obs["Sigma_T"].values, zh)
        mP, sP = eof_predict(eof["P"], zo, Po, obs["sig_lnP"].values, zh)
        P["eof"] = (mT, mP, sT, sP)
    nn_needed = [m for m in methods if m.endswith(("_zs", "_ft", "_rft"))]
    if nn_needed:
        gl, gh = zh.min(), zh.max()
        colloc = np.vstack([densify(prof, CFG["N_COLLOC_GAP"], rng, gl, gh),
                            densify(prof, CFG["N_COLLOC_GAP"] // 2, rng)])
        for fam in base:
            key = fam.lower()
            members = base[fam][:seeds_limit] if seeds_limit else base[fam]
            zs, ft, rft = [], [], []
            for _, b in members:
                if f"{key}_zs" in methods:
                    zs.append(predict(b, hid))
                if f"{key}_ft" in methods:
                    lam = LAM[fam] if lam_override is None or fam == "MLP" else lam_override
                    ft.append(predict(finetune(b, obs, colloc, lam), hid))
                if f"{key}_rft" in methods:
                    h = (hp_override or hp or HP)[fam]
                    rft.append(predict(finetune_reg(b, obs, colloc, h["lam"], h["alpha"], gl, gh, h.get("es", True)), hid))
            for tag, lst in ((f"{key}_zs", zs), (f"{key}_ft", ft), (f"{key}_rft", rft)):
                if lst:
                    A = np.array(lst)   # (members, 2, n)
                    sd = A.std(axis=0, ddof=1) if len(lst) > 1 else np.full(A.shape[1:], np.nan)
                    P[tag] = (A[:, 0].mean(0), A[:, 1].mean(0), sd[0], sd[1])
    rows = []
    for k, (Tp, Pp, sT, sP) in P.items():
        rows.append(pd.DataFrame(dict(method=NAMES[k], Z_km=zh, T_ref=hid["Temperature"].values,
                                      lnP_ref=hid["lnP"].values, sig_T=hid["Sigma_T"].values,
                                      sig_lnP=hid["sig_lnP"].values, T_pred=Tp, lnP_pred=Pp,
                                      T_std=np.nan if sT is None else sT, lnP_std=np.nan if sP is None else sP)))
    return pd.concat(rows, ignore_index=True)

def run_benchmark(cases, df, base, rf, cache_name, **kw):
    cache = RES / f"{cache_name}.pkl"
    done = pd.read_pickle(cache) if cache.exists() else pd.DataFrame()
    done_keys = set(done["case_id"].unique()) if len(done) else set()
    parts, t0 = [done], time.time()
    todo = [c for c in cases if "|".join(map(str, c)) not in done_keys]
    print(f"{cache_name}: {len(cases) - len(todo)} cached, {len(todo)} to run")
    for i, case in enumerate(todo):
        pid, size, loc = case[:3]
        prof = df[df["Profile_ID"] == pid].sort_values("Z_km").reset_index(drop=True)
        gap = make_gap(prof["Z_km"].values, size, loc)
        extra = dict(case[3]) if len(case) > 3 else {}
        out = run_case(prof, gap, base, rf, rng_seed=case_seed(case[:3]), **kw, **extra)
        out["case_id"] = "|".join(map(str, case)); out["Profile_ID"] = pid
        out["gap_km"] = size; out["location"] = loc
        for k, v in extra.items():
            out[k] = v
        parts.append(out)
        if (i + 1) % 10 == 0 or i + 1 == len(todo):
            pd.concat(parts, ignore_index=True).to_pickle(cache)
            el = time.time() - t0
            print(f"  {i + 1}/{len(todo)} cases  {el:.0f}s  (~{el / (i + 1) * (len(todo) - i - 1):.0f}s left)")
    return pd.concat(parts, ignore_index=True)

## 7a. Hyper-parameter selection on validation profiles
Regularisation strength α (both families), whether to use early stopping (both), and physics weight λ (PINN) for the
regularised fine-tuning are chosen on
**validation profiles only**, using the gap configurations in `TUNE_CONFIGS` and the first pre-training seed.
Selection rule (fixed in advance): minimise $\overline{\text{T MAE}}/\min(\overline{\text{T MAE}})+
\overline{\text{P error}}/\min(\overline{\text{P error}})$ over the grid, i.e. equal weight on temperature and pressure.
Test profiles are never used for any choice.

In [ ]:
HP_FILE = OUT / "selected_hyperparameters.json"
TUNE_CASES = [(pid, s, loc) for pid in VAL_IDS for (s, loc) in CFG["TUNE_CONFIGS"]
              if make_gap(val_df.loc[val_df.Profile_ID == pid, "Z_km"].values, s, loc) is not None]
_tune_rows = []

def _quick_metrics(pw):
    e = pw.assign(aT=(pw.T_pred - pw.T_ref).abs(), aP=100 * (np.exp(pw.lnP_pred - pw.lnP_ref) - 1).abs())
    pc = e.groupby(["Profile_ID", "case_id"])[["aT", "aP"]].mean().groupby("Profile_ID").mean()
    return pc["aT"].mean(), pc["aP"].mean()

if HP_FILE.exists():
    HP = json.load(open(HP_FILE)); print("Loaded selected hyper-parameters:", HP)
else:
    HP = {}
    for fam in ["MLP", "PINN"]:
        lams = [0.0] if fam == "MLP" else CFG["TUNE_LAMBDAS"]
        for lam in lams:
            for alpha in CFG["TUNE_ALPHAS"]:
                for es in CFG["TUNE_ES"]:
                    hpo = {fam: dict(lam=lam, alpha=alpha, es=es)}
                    pw = run_benchmark(TUNE_CASES, val_df, {fam: BASE[fam]}, None,
                                       f"tune_{fam}_lam{lam}_alpha{alpha}_es{int(es)}", methods=[f"{fam.lower()}_rft"],
                                       seeds_limit=1, hp_override=hpo)
                    mT, mP = _quick_metrics(pw)
                    _tune_rows.append(dict(family=fam, lam=lam, alpha=alpha, es=es, T_MAE=mT, P_relMAE=mP))
    TUNE = pd.DataFrame(_tune_rows)
    TUNE["score"] = TUNE.groupby("family")["T_MAE"].transform(lambda x: x / x.min()) + \
                    TUNE.groupby("family")["P_relMAE"].transform(lambda x: x / x.min())
    TUNE.to_csv(TAB / "tuning_validation.csv", index=False)
    display(TUNE.round(4))
    for fam, g in TUNE.groupby("family"):
        r = g.loc[g["score"].idxmin()]
        HP[fam] = dict(lam=float(r["lam"]), alpha=float(r["alpha"]), es=bool(r["es"]))
    json.dump(HP, open(HP_FILE, "w"), indent=2)
print("Selected (validation):", HP)

### 7b. Run the main benchmark on test profiles

In [ ]:
PW = run_benchmark(CASES, test_df, BASE, RF_MAIN, "pointwise_main", eof=EOF_MAIN)
print(PW.groupby("method").size())

## 8. Metrics, aggregation and statistics

Per case and method (over hidden levels):
- **T**: MAE, RMSE, bias (K); fraction of levels with $|e_T| \le 2\sigma_T$ (within the retrieval's own uncertainty).
- **P**: mean relative error (%) $= 100\,\overline{|e^{\Delta\ln P}-1|}$.
- **n**: mean relative error (%) of the derived $n=P/(k_BT)$.
- **Hydrostatic residual**: RMS of $r_{\rm hyd}$ evaluated by finite differences on the predicted profile
  (same operator for every method, compared with the reference-data floor).
- **Uncertainty** (GP, ensembles): empirical coverage of central intervals at 50/80/90/95 %, using
  $\sigma_{\rm tot}=\sqrt{\sigma_{\rm pred}^2+\sigma_{\rm ret}^2}$ because the reference itself is uncertain;
  Spearman correlation between $\sigma_{\rm pred}$ and $|e|$.

**Statistics.** Cases from the same profile are not independent, so every comparison first averages over the cases of
a profile, then uses the profile as the unit: 95 % CIs from a bootstrap over profiles, and paired
Wilcoxon signed-rank tests with Holm correction across competitors.

In [ ]:
COV_LEVELS = [0.5, 0.8, 0.9, 0.95]

def case_metrics(g):
    g = g.sort_values("Z_km")
    eT = g["T_pred"].values - g["T_ref"].values
    elnP = g["lnP_pred"].values - g["lnP_ref"].values
    n_ratio = np.exp(elnP) * g["T_ref"].values / g["T_pred"].values
    d = dict(T_MAE=np.mean(np.abs(eT)), T_RMSE=np.sqrt(np.mean(eT ** 2)), T_bias=np.mean(eT),
             T_within_2sig=np.mean(np.abs(eT) <= 2 * g["sig_T"].values),
             P_relMAE=100 * np.mean(np.abs(np.exp(elnP) - 1)), n_relMAE=100 * np.mean(np.abs(n_ratio - 1)),
             Hydro_RMS=np.sqrt(np.mean(hydro_residual_np(g["Z_km"].values, g["T_pred"].values,
                                                         g["lnP_pred"].values) ** 2)) if len(g) >= 3 else np.nan)
    if g["T_std"].notna().all():
        sT = np.sqrt(g["T_std"].values ** 2 + g["sig_T"].values ** 2)
        sP = np.sqrt(g["lnP_std"].values ** 2 + g["sig_lnP"].values ** 2)
        for q in COV_LEVELS:
            zq = stats.norm.ppf(0.5 + q / 2)
            d[f"T_cov{int(q * 100)}"] = np.mean(np.abs(eT) <= zq * sT)
            d[f"P_cov{int(q * 100)}"] = np.mean(np.abs(elnP) <= zq * sP)
        d["T_std_mean"] = g["T_std"].mean()
    return pd.Series(d)

def compute_metrics(pw, keys=("case_id", "Profile_ID", "gap_km", "location", "method")):
    keys = [k for k in keys if k in pw.columns]
    rows = []
    for k, g in pw.groupby(keys):
        k = k if isinstance(k, tuple) else (k,)
        rows.append({**dict(zip(keys, k)), **case_metrics(g).to_dict()})
    return pd.DataFrame(rows)

def boot_ci(x, n=CFG["N_BOOT"], seed=0):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if len(x) < 2:
        return (np.nan, np.nan)
    b = np.random.default_rng(seed).choice(x, (n, len(x))).mean(1)
    return tuple(np.percentile(b, [2.5, 97.5]))

def summarize(M, metrics, by=("method",)):
    """Profile-level means, then mean and bootstrap CI over profiles."""
    by = list(by)
    per_prof = M.groupby(by + ["Profile_ID"])[metrics].mean().reset_index()
    rows = []
    for key, g in per_prof.groupby(by):
        key = key if isinstance(key, tuple) else (key,)
        r = dict(zip(by, key)); r["n_profiles"] = g["Profile_ID"].nunique()
        for m in metrics:
            lo, hi = boot_ci(g[m].values)
            r[m] = g[m].mean(); r[m + "_lo"] = lo; r[m + "_hi"] = hi
        rows.append(r)
    return pd.DataFrame(rows)

def paired_tests(M, metric, ours=OURS, by=None):
    """Positive mean_diff = ours has lower error."""
    out = []
    groups = [(None, M)] if by is None else list(M.groupby(by))
    for gkey, G in groups:
        pp = G.groupby(["method", "Profile_ID"])[metric].mean().unstack(0)
        if ours not in pp:
            continue
        for other in [c for c in pp.columns if c != ours]:
            d = (pp[other] - pp[ours]).dropna()
            if len(d) < 3:
                continue
            if np.allclose(d.values, 0):
                p = 1.0                       # identical predictions (e.g. selection chose the same settings)
            else:
                try:
                    p = stats.wilcoxon(d.values).pvalue
                except ValueError:
                    p = 1.0
            lo, hi = boot_ci(d.values)
            out.append(dict(group=gkey, metric=metric, competitor=other, n_profiles=len(d),
                            mean_diff=d.mean(), ci_lo=lo, ci_hi=hi,
                            rel_improvement_pct=100 * d.mean() / pp[other].loc[d.index].mean(),
                            ours_better_frac=(d > 0).mean(), p_wilcoxon=p))
    T = pd.DataFrame(out)
    if len(T):   # Holm correction within each (group, metric)
        T["p_holm"] = np.nan
        for _, idx in T.groupby(["group", "metric"], dropna=False).groups.items():
            p = T.loc[idx, "p_wilcoxon"].values; o = np.argsort(p); m = len(p)
            adj = np.maximum.accumulate(np.minimum(1, (m - np.arange(m)) * p[o]))
            T.loc[np.array(idx)[o], "p_holm"] = adj
        T["significant"] = T["p_holm"] < CFG["ALPHA"]
    return T

M_MAIN = compute_metrics(PW)
M_MAIN.to_csv(TAB / "metrics_per_case_main.csv", index=False)
POINT_METRICS = ["T_MAE", "T_RMSE", "T_bias", "T_within_2sig", "P_relMAE", "n_relMAE", "Hydro_RMS"]
S_OVERALL = summarize(M_MAIN, POINT_METRICS)
S_BY_LOC = summarize(M_MAIN, POINT_METRICS, by=("location", "method"))
S_BY_GAP = summarize(M_MAIN, POINT_METRICS, by=("location", "gap_km", "method"))
SIG = pd.concat([paired_tests(M_MAIN, m) for m in ["T_MAE", "P_relMAE", "n_relMAE"]], ignore_index=True)
SIG_LOC = pd.concat([paired_tests(M_MAIN, m, by="location") for m in ["T_MAE", "P_relMAE", "n_relMAE"]],
                    ignore_index=True)
for name, df_ in [("summary_overall", S_OVERALL), ("summary_by_location", S_BY_LOC), ("summary_by_gap", S_BY_GAP),
                  ("significance_overall", SIG), ("significance_by_location", SIG_LOC)]:
    df_.to_csv(TAB / f"{name}.csv", index=False)
show = S_OVERALL.set_index("method")[["n_profiles", "T_MAE", "T_RMSE", "T_within_2sig", "P_relMAE", "n_relMAE", "Hydro_RMS"]]
display(show.sort_values("T_MAE").round(3))
print(f"Reference-data hydrostatic floor (median RMS): {REF_HYDRO_FLOOR:.4f}")
display(SIG.round(4))

In [ ]:
# --- Uncertainty quantification summary ---
UQ_METHODS = [NAMES[k] for k in ("gp", "eof", "mlp_rft", "pinn_rft", "mlp_ft", "pinn_ft", "mlp_zs", "pinn_zs")]
uq_cols = [c for c in M_MAIN.columns if "_cov" in c]
UQ = summarize(M_MAIN[M_MAIN["method"].isin(UQ_METHODS)], uq_cols + ["T_std_mean"])
rho_rows = []
for m in UQ_METHODS:
    g = PW[(PW["method"] == m) & PW["T_std"].notna()]
    if len(g) > 10 and g["T_std"].nunique() > 1 and g["lnP_std"].nunique() > 1:
        rT = stats.spearmanr(g["T_std"], np.abs(g["T_pred"] - g["T_ref"])).correlation
        rP = stats.spearmanr(g["lnP_std"], np.abs(g["lnP_pred"] - g["lnP_ref"])).correlation
        rho_rows.append(dict(method=m, spearman_T=rT, spearman_lnP=rP))
UQ = UQ.merge(pd.DataFrame(rho_rows), on="method", how="left")
UQ.to_csv(TAB / "uncertainty_summary.csv", index=False)
display(UQ.set_index("method")[[c for c in UQ.columns if c.endswith(("cov50", "cov80", "cov90", "cov95"))
                                 or c.startswith("spearman") or c == "T_std_mean"]].round(3))

## 8b. Conformal calibration
Raw ensemble and GP intervals are over-confident. Split conformal prediction fixes the coverage using **validation
profiles**: for each method and gap location, the normalised scores $s=|e|/\sigma_{\rm tot}$ are collected on
validation gap cases, and the calibrated half-width at level $q$ is $\hat c_q\,\sigma_{\rm tot}$ with
$\hat c_q$ the $\lceil (n+1)q\rceil/n$ empirical quantile of $s$ (Mondrian calibration by location).
Levels within one profile are correlated, so the finite-sample guarantee is approximate; empirical test coverage is
reported.

In [ ]:
CAL_METHODS = ["gp", "eof", "mlp_rft", "pinn_rft"]
QGRID = np.round(np.arange(0.05, 0.951, 0.05), 2)
CAL_CASES = [(pid, s, loc) for pid in VAL_IDS for s in CFG["CAL_GAP_SIZES_KM"] for loc in CFG["GAP_LOCATIONS"]
             if make_gap(val_df.loc[val_df.Profile_ID == pid, "Z_km"].values, s, loc) is not None]
PW_CAL = run_benchmark(CAL_CASES, val_df, BASE, None, "pointwise_calibration", methods=CAL_METHODS, eof=EOF_MAIN)

def _scores(g):
    sT = np.sqrt(g.T_std ** 2 + g.sig_T ** 2); sP = np.sqrt(g.lnP_std ** 2 + g.sig_lnP ** 2)
    return (np.abs(g.T_pred - g.T_ref) / sT).values, (np.abs(g.lnP_pred - g.lnP_ref) / sP).values

CONF_Q = {}   # (method, location, var, q) -> calibrated score quantile
for (m, loc), g in PW_CAL[PW_CAL.T_std.notna()].groupby(["method", "location"]):
    for var, r in zip(("T", "P"), _scores(g)):
        n = len(r)
        for q in QGRID:
            CONF_Q[(m, loc, var, q)] = np.quantile(r, min(1.0, np.ceil((n + 1) * q) / n))

def conformal_coverage(pw):
    rows = []
    for (cid, pid, loc, m), g in pw[pw.method.isin(set(k[0] for k in CONF_Q))].groupby(
            ["case_id", "Profile_ID", "location", "method"]):
        if g.T_std.isna().any() or (m, loc, "T", 0.95) not in CONF_Q:
            continue
        rT, rP = _scores(g); d = dict(case_id=cid, Profile_ID=pid, location=loc, method=m)
        for q in COV_LEVELS:
            d[f"T_ccov{int(q * 100)}"] = np.mean(rT <= CONF_Q[(m, loc, "T", q)])
            d[f"P_ccov{int(q * 100)}"] = np.mean(rP <= CONF_Q[(m, loc, "P", q)])
        rows.append(d)
    return pd.DataFrame(rows)

CC = conformal_coverage(PW)
ccols = [c for c in CC.columns if "_ccov" in c]
UQC = summarize(CC, ccols)
UQ = UQ.merge(UQC[["method"] + ccols], on="method", how="left")
UQ.to_csv(TAB / "uncertainty_summary.csv", index=False)
display(UQ.set_index("method")[[c for c in UQ.columns if c.endswith(("cov90", "cov95"))]].round(3))

## 9. Additional experiments

**E2 — Noise robustness.** Gaussian noise of $k\,\sigma_{\rm ret}$ (k ∈ `NOISE_MULTIPLIERS`) is added to the
*observed* levels only; errors are measured against the clean reference.

**E3 — Physics-weight sensitivity.** The plain PINN fine-tuning weight λ is varied on test profiles (pre-training
fixed at the default λ); λ = 0 is fine-tuning the PINN-pre-trained network without physics. This is a reported
sensitivity analysis only; λ for the main method was selected on validation profiles (§7a).

**E4 — Mars-year hold-out.** Pre-train on two Mars years, test on profiles of the third. Each Mars year in this
dataset is one observing campaign covering a different $L_s$ range and latitude band (MY 29: ≈13°S–15°N; MY 30 and
31: ≈47–70°N), so this tests transfer to an unseen campaign, season and latitude band together — a harder setting
than E1.

In [ ]:
NOISE_METHODS = ["lin", "pchip", "hydro", "gp", "eof", "mlp_rft", "pinn_rft"]
M_NOISE = None
if CFG["RUN_NOISE"]:
    noise_cases = [(pid, s, loc, (("noise_mult", k),)) for pid in TEST_IDS for (s, loc) in CFG["NOISE_CASES"]
                   for k in CFG["NOISE_MULTIPLIERS"]
                   if make_gap(test_df.loc[test_df.Profile_ID == pid, "Z_km"].values, s, loc) is not None]
    PW_NOISE = run_benchmark(noise_cases, test_df, BASE, None, "pointwise_noise",
                             methods=NOISE_METHODS, seeds_limit=CFG["NOISE_SEEDS"], eof=EOF_MAIN)
    M_NOISE = compute_metrics(PW_NOISE, keys=("case_id", "Profile_ID", "gap_km", "location", "noise_mult", "method"))
    S_NOISE = summarize(M_NOISE, ["T_MAE", "P_relMAE", "n_relMAE"], by=("noise_mult", "method"))
    S_NOISE.to_csv(TAB / "summary_noise.csv", index=False)
    display(S_NOISE.pivot(index="method", columns="noise_mult", values="T_MAE").round(3))

In [ ]:
M_LAM = None
if CFG["RUN_LAMBDA_SWEEP"]:
    parts = []
    for lam in CFG["LAMBDAS"]:
        lam_cases = [(pid, s, loc, (("lam_override", lam),)) for pid in TEST_IDS for (s, loc) in CFG["LAMBDA_CASES"]
                     if make_gap(test_df.loc[test_df.Profile_ID == pid, "Z_km"].values, s, loc) is not None]
        parts.append(run_benchmark(lam_cases, test_df, {"PINN": BASE["PINN"]}, None, f"pointwise_lambda_{lam}",
                                   methods=["pinn_ft"], seeds_limit=1))
    PW_LAM = pd.concat(parts, ignore_index=True)
    M_LAM = compute_metrics(PW_LAM, keys=("case_id", "Profile_ID", "gap_km", "location", "lam_override", "method"))
    S_LAM = summarize(M_LAM, ["T_MAE", "P_relMAE", "n_relMAE", "Hydro_RMS"], by=("location", "lam_override"))
    S_LAM.to_csv(TAB / "summary_lambda.csv", index=False)
    display(S_LAM.round(4))

In [ ]:
M_HOLD = None
if CFG["RUN_YEAR_HOLDOUT"]:
    HOLD_METHODS = ["hydro", "gp", "eof", "rf", "mlp_zs", "pinn_zs", "mlp_rft", "pinn_rft"]
    seeds_h = CFG["SEEDS"][:CFG["HOLDOUT_SEEDS"]]
    parts = []
    for my in sorted(meta["Mars_Year"].unique()):
        test_ids = sorted(meta.index[meta["Mars_Year"] == my])[:CFG["HOLDOUT_MAX_TEST"]]
        rest = meta.index[meta["Mars_Year"] != my].values
        if len(test_ids) < 3 or len(rest) < 10:
            continue
        rng = np.random.default_rng(CFG["SPLIT_SEED"]); rest = rng.permutation(rest)
        n_va = max(1, int(0.15 * len(rest)))
        tr_h = lv[lv["Profile_ID"].isin(rest[n_va:])]; va_h = lv[lv["Profile_ID"].isin(rest[:n_va])]
        te_h = lv[lv["Profile_ID"].isin(test_ids)]
        norm_h = compute_norm(tr_h)
        base_h = {fam: [(s, pretrain(tr_h, va_h, fam, LAM[fam], s, tag=f"holdMY{int(my)}", norm=norm_h))
                        for s in seeds_h] for fam in LAM}
        rf_h = fit_rf(tr_h); eof_h = fit_eof_pair(tr_h)
        cases_h = [(pid, s, loc) for pid in test_ids for (s, loc) in CFG["HOLDOUT_CASES"]
                   if make_gap(te_h.loc[te_h.Profile_ID == pid, "Z_km"].values, s, loc) is not None]
        pw = run_benchmark(cases_h, te_h, base_h, rf_h, f"pointwise_holdout_MY{int(my)}", methods=HOLD_METHODS, eof=eof_h)
        pw["holdout_MY"] = int(my); parts.append(pw)
    if parts:
        PW_HOLD = pd.concat(parts, ignore_index=True)
        M_HOLD = compute_metrics(PW_HOLD, keys=("case_id", "Profile_ID", "gap_km", "location", "holdout_MY", "method"))
        S_HOLD = summarize(M_HOLD, ["T_MAE", "P_relMAE", "n_relMAE"], by=("holdout_MY", "method"))
        S_HOLD.to_csv(TAB / "summary_year_holdout.csv", index=False)
        SIG_HOLD = pd.concat([paired_tests(M_HOLD, m) for m in ["T_MAE", "P_relMAE"]], ignore_index=True)
        SIG_HOLD.to_csv(TAB / "significance_year_holdout.csv", index=False)
        display(S_HOLD.pivot(index="method", columns="holdout_MY", values="T_MAE").round(3))

## 10. Figures and tables
All figures are saved as PNG (300 dpi) and PDF in `results/figures/`; tables as CSV and LaTeX in `results/tables/`.

In [ ]:
ORDER = [NAMES[k] for k in ALL_METHODS]
COLORS = dict(zip(ORDER, ["#9e9e9e", "#616161", "#8d6e63", "#7cb342", "#9c27b0", "#26a69a",
                          "#f4a582", "#92c5de", "#e7a3a0", "#9ec3e6", "#d6604d", "#2166ac"]))

# --- Figure 2: example reconstructions ---
ex_methods = [NAMES[k] for k in ("lin", "hydro", "eof", "mlp_rft", "pinn_rft")]
_have = set(PW["case_id"])
_pid = next((p for p in TEST_IDS if f"{p}|10|middle" in _have and f"{p}|10|upper" in _have), None)
ex_cases = [] if _pid is None else [(_pid, 10, "middle"), (_pid, 10, "upper")]
if ex_cases:
    fig, axes = plt.subplots(len(ex_cases), 3, figsize=(14, 4.6 * len(ex_cases)), squeeze=False)
    for r, case in enumerate(ex_cases):
        prof = test_df[test_df.Profile_ID == case[0]].sort_values("Z_km")
        g = PW[PW.case_id == "|".join(map(str, case))]
        zg = g["Z_km"].unique()
        ax = axes[r]
        ax[0].plot(prof["Temperature"], prof["Z_km"], "k-", lw=1.8, label="MRO RSS retrieval")
        ax[0].fill_betweenx(prof["Z_km"], prof["Temperature"] - 2 * prof["Sigma_T"],
                            prof["Temperature"] + 2 * prof["Sigma_T"], color="k", alpha=0.08, label=r"retrieval $\pm2\sigma$")
        for a in ax:
            a.axhspan(zg.min(), zg.max(), color="gold", alpha=0.15)
        for m in ex_methods:
            gm = g[g.method == m].sort_values("Z_km")
            ax[0].plot(gm["T_pred"], gm["Z_km"], color=COLORS[m], lw=1.6, label=m)
            ax[1].plot(100 * (np.exp(gm["lnP_pred"] - gm["lnP_ref"]) - 1), gm["Z_km"], color=COLORS[m], lw=1.6)
            nr = np.exp(gm["lnP_pred"] - gm["lnP_ref"]) * gm["T_ref"] / gm["T_pred"]
            ax[2].plot(100 * (nr - 1), gm["Z_km"], color=COLORS[m], lw=1.6)
            if m == OURS and gm["T_std"].notna().all():
                ax[0].fill_betweenx(gm["Z_km"], gm["T_pred"] - 1.96 * gm["T_std"], gm["T_pred"] + 1.96 * gm["T_std"],
                                    color=COLORS[m], alpha=0.2)
        for a in ax[1:]:
            a.axvline(0, color="k", lw=0.8)
        ax[0].set_xlabel("T (K)"); ax[1].set_xlabel("P error (%)"); ax[2].set_xlabel("n error (%)")
        ax[0].set_ylabel(f"Z (km) — {case[1]} km {case[2]} gap")
        ax[0].set_ylim(prof["Z_km"].min(), prof["Z_km"].max())
        for a in ax[1:]:
            a.set_ylim(zg.min() - 1, zg.max() + 1)
        if r == 0:
            ax[0].legend(fontsize=7, loc="best")
    fig.suptitle(f"Profile {ex_cases[0][0]}: reconstruction inside the gap (shaded)", y=1.0)
    fig.tight_layout(); savefig(fig, "fig02_example_reconstructions"); plt.show()

In [ ]:
# --- Figure 3: error vs gap size, by location ---
fig, axes = plt.subplots(3, len(CFG["GAP_LOCATIONS"]), figsize=(5 * len(CFG["GAP_LOCATIONS"]), 11), squeeze=False)
for j, loc in enumerate(CFG["GAP_LOCATIONS"]):
    for i, (met, lab) in enumerate([("T_MAE", "T MAE (K)"), ("P_relMAE", "P error (%)"), ("n_relMAE", "n error (%)")]):
        ax = axes[i, j]
        for m in [o for o in ORDER if o not in (NAMES["mlp_ft"], NAMES["pinn_ft"])]:
            s = S_BY_GAP[(S_BY_GAP.location == loc) & (S_BY_GAP.method == m)].sort_values("gap_km")
            if len(s):
                ax.errorbar(s["gap_km"], s[met], yerr=[s[met] - s[met + "_lo"], s[met + "_hi"] - s[met]],
                            color=COLORS[m], marker="o", ms=4, capsize=2, lw=2.2 if m == OURS else 1.2, label=m)
        ax.set_yscale("log"); ax.set_xlabel("Gap size (km)"); ax.set_ylabel(lab)
        if i == 0:
            ax.set_title(f"{loc} gap")
axes[0, 0].legend(fontsize=7)
fig.tight_layout(); savefig(fig, "fig03_error_vs_gap_size"); plt.show()

# --- Figure 4: hydrostatic residual ---
fig, ax = plt.subplots(figsize=(10, 4))
data = [M_MAIN.loc[M_MAIN.method == m, "Hydro_RMS"].dropna().values for m in ORDER]
try:
    bp = ax.boxplot(data, tick_labels=ORDER, showfliers=False, patch_artist=True)
except TypeError:   # matplotlib < 3.9
    bp = ax.boxplot(data, labels=ORDER, showfliers=False, patch_artist=True)
for patch, m in zip(bp["boxes"], ORDER):
    patch.set_facecolor(COLORS[m]); patch.set_alpha(0.7)
ax.axhline(REF_HYDRO_FLOOR, color="k", ls="--", lw=1, label="reference-data floor")
ax.set_yscale("log"); ax.set_ylabel(r"RMS hydrostatic residual $r_{hyd}$ (dimensionless)")
ax.tick_params(axis="x", rotation=30); ax.legend()
fig.tight_layout(); savefig(fig, "fig04_hydrostatic_residual"); plt.show()

In [ ]:
# --- Figure 5: calibration and uncertainty-error relation ---
nominal = np.linspace(0.05, 0.95, 19)
fig, ax = plt.subplots(1, 3, figsize=(15, 4.3))
nominal = QGRID
for m in [NAMES["gp"], NAMES["eof"], NAMES["mlp_rft"], NAMES["pinn_rft"]]:
    g = PW[(PW.method == m) & PW.T_std.notna()]
    if not len(g):
        continue
    eT = np.abs(g.T_pred - g.T_ref); sT = np.sqrt(g.T_std ** 2 + g.sig_T ** 2)
    eP = np.abs(g.lnP_pred - g.lnP_ref); sP = np.sqrt(g.lnP_std ** 2 + g.sig_lnP ** 2)
    ax[0].plot(nominal, [np.mean(eT <= stats.norm.ppf(0.5 + q / 2) * sT) for q in nominal], "o--", ms=2, lw=1, alpha=0.6, color=COLORS[m])
    ax[1].plot(nominal, [np.mean(eP <= stats.norm.ppf(0.5 + q / 2) * sP) for q in nominal], "o--", ms=2, lw=1, alpha=0.6, color=COLORS[m])
    locs = g["location"].values; rT = (eT / sT).values; rP = (eP / sP).values
    for a, r, var in ((ax[0], rT, "T"), (ax[1], rP, "P")):
        cov = [np.mean([ri <= CONF_Q.get((m, l, var, q), np.inf) for ri, l in zip(r, locs)]) for q in nominal]
        a.plot(nominal, cov, "o-", ms=3, color=COLORS[m], label=f"{m} (conformal)")
    bins = pd.qcut(g.T_std, 5, duplicates="drop")
    b = pd.DataFrame(dict(s=g.T_std, e=eT)).groupby(bins, observed=True).mean()
    ax[2].plot(b["s"], b["e"], "o-", color=COLORS[m], label=m)
for a, t in zip(ax[:2], ["Temperature", "ln P"]):
    a.plot([0, 1], [0, 1], "k--", lw=1); a.set_xlabel("Nominal coverage"); a.set_ylabel("Empirical coverage")
    a.set_title(f"Calibration: {t}  (dashed = raw)"); a.legend(fontsize=7)
lim = ax[2].get_xlim(); ax[2].plot(lim, lim, "k:", lw=1)
ax[2].set_xlabel(r"Predicted $\sigma_T$ (K), quintile mean"); ax[2].set_ylabel("Mean |T error| (K)")
ax[2].set_title("Does uncertainty rank errors?"); ax[2].legend(fontsize=8)
fig.tight_layout(); savefig(fig, "fig05_uncertainty_calibration"); plt.show()

# --- Figure 6: paired improvement of the PINN over each competitor ---
if len(SIG):
    mets = ["T_MAE", "P_relMAE", "n_relMAE"]
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    for a, met in zip(ax, mets):
        s = SIG[SIG.metric == met].set_index("competitor").reindex([o for o in ORDER if o != OURS]).dropna(subset=["mean_diff"])
        y = np.arange(len(s))
        a.errorbar(s["mean_diff"], y, xerr=[s["mean_diff"] - s["ci_lo"], s["ci_hi"] - s["mean_diff"]],
                   fmt="o", color="#2166ac", capsize=3)
        for yi, (_, r) in zip(y, s.iterrows()):
            a.text(r["ci_hi"], yi, "  *" if r["significant"] else "", va="center")
        a.axvline(0, color="k", lw=1); a.set_yticks(y); a.set_yticklabels(s.index)
        a.set_title(f"{met}: competitor − PINN (>0 favours PINN)", fontsize=9)
    fig.tight_layout(); savefig(fig, "fig06_paired_improvement"); plt.show()

In [ ]:
# --- Figures 7-9: noise, lambda, Mars-year hold-out; Figure 10: training curves ---
if M_NOISE is not None:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for m in S_NOISE.method.unique():
        s = S_NOISE[S_NOISE.method == m].sort_values("noise_mult")
        for a, met in zip(ax, ["T_MAE", "P_relMAE"]):
            a.errorbar(s.noise_mult, s[met], yerr=[s[met] - s[met + "_lo"], s[met + "_hi"] - s[met]],
                       marker="o", capsize=2, color=COLORS[m], label=m)
    ax[0].set_ylabel("T MAE (K)"); ax[1].set_ylabel("P error (%)")
    for a in ax:
        a.set_xlabel(r"Added noise ($\times\sigma_{ret}$)"); a.set_yscale("log")
    ax[0].legend(fontsize=7); fig.tight_layout(); savefig(fig, "fig07_noise_robustness"); plt.show()

if M_LAM is not None:
    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for loc in S_LAM.location.unique():
        s = S_LAM[S_LAM.location == loc].sort_values("lam_override")
        x = s.lam_override.replace(0, CFG["LAMBDAS"][1] / 10 if len(CFG["LAMBDAS"]) > 1 else 1e-3)
        for a, met in zip(ax, ["T_MAE", "P_relMAE", "Hydro_RMS"]):
            a.errorbar(x, s[met], yerr=[s[met] - s[met + "_lo"], s[met + "_hi"] - s[met]], marker="o", capsize=2, label=f"{loc}")
            a.set_xscale("log"); a.set_xlabel(r"$\lambda$ (fine-tuning; leftmost point = 0)"); a.set_ylabel(met)
    ax[2].axhline(REF_HYDRO_FLOOR, color="k", ls="--", lw=1); ax[0].legend()
    fig.tight_layout(); savefig(fig, "fig08_lambda_sensitivity"); plt.show()

if M_HOLD is not None:
    piv = S_HOLD.pivot(index="method", columns="holdout_MY", values="T_MAE")
    piv = piv.reindex([o for o in ORDER if o in piv.index])
    fig, ax = plt.subplots(figsize=(10, 4))
    w = 0.8 / len(piv.columns)
    for k, my in enumerate(piv.columns):
        ax.bar(np.arange(len(piv)) + k * w, piv[my], w, label=f"held-out MY {my}")
    ax.set_xticks(np.arange(len(piv)) + 0.4 - w / 2); ax.set_xticklabels(piv.index, rotation=30, ha="right")
    ax.set_ylabel("T MAE (K)"); ax.legend(); fig.tight_layout(); savefig(fig, "fig09_year_holdout"); plt.show()

hist_files = sorted(RES.glob("history_main_*_seed*.csv"))
if hist_files:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    for f in hist_files:
        h = pd.read_csv(f); fam = "PINN" if "_PINN_" in f.name else "MLP"
        c = COLORS[NAMES["pinn_ft"] if fam == "PINN" else NAMES["mlp_ft"]]
        ax[0].plot(h.epoch, h.val_data, color=c, alpha=0.6, lw=1)
        if fam == "PINN":
            ax[1].plot(h.epoch, h.train_phys, color=c, alpha=0.6, lw=1)
    ax[0].set_yscale("log"); ax[0].set_title("Validation data loss (blue: PINN, red: MLP)"); ax[0].set_xlabel("epoch")
    ax[1].set_yscale("log"); ax[1].set_title("PINN physics loss during pre-training"); ax[1].set_xlabel("epoch")
    fig.tight_layout(); savefig(fig, "fig10_training_curves"); plt.show()

In [ ]:
# --- LaTeX tables ---
def fmt_ci(r, m, d=2):
    return f"{r[m]:.{d}f} [{r[m + '_lo']:.{d}f}, {r[m + '_hi']:.{d}f}]"

def latex_table(df, path, caption, label):
    esc = lambda s: (str(s).replace("%", r"\%").replace("_", r"\_").replace("&", r"\&")
                     .replace("σ", r"$\sigma$").replace("Δ", r"$\Delta$").replace("−", "-"))
    cols = list(df.columns)
    lines = [r"\begin{table}[t]", r"\centering", r"\small", rf"\caption{{{esc(caption)}}}", rf"\label{{{label}}}",
             r"\begin{tabular}{l" + "c" * len(cols) + "}", r"\toprule",
             " & ".join([esc(df.index.name or "")] + [esc(c) for c in cols]) + r" \\", r"\midrule"]
    for idx, row in df.iterrows():
        lines.append(" & ".join([esc(idx)] + [esc(v) for v in row.values]) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    Path(path).write_text("\n".join(lines))

main = S_OVERALL.set_index("method").reindex([o for o in ORDER if o in set(S_OVERALL.method)])
tab1 = pd.DataFrame({
    "T MAE (K)": [fmt_ci(r, "T_MAE") for _, r in main.iterrows()],
    "within 2σ_T": [f"{r['T_within_2sig']:.2f}" for _, r in main.iterrows()],
    "P error (%)": [fmt_ci(r, "P_relMAE") for _, r in main.iterrows()],
    "n error (%)": [fmt_ci(r, "n_relMAE") for _, r in main.iterrows()],
    "Hydro. residual": [f"{r['Hydro_RMS']:.3g}" for _, r in main.iterrows()],
}, index=main.index); tab1.index.name = "Method"
latex_table(tab1, TAB / "table1_main_results.tex",
            f"Reconstruction error over all gap cases ({int(main['n_profiles'].max())} test profiles; mean and 95% bootstrap CI over profiles). Reference hydrostatic floor = {REF_HYDRO_FLOOR:.3g}.",
            "tab:main")
display(tab1)

if len(SIG):
    tab2 = SIG.copy()
    tab2["Δ (competitor − PINN)"] = [f"{a:.3g} [{b:.3g}, {c:.3g}]" for a, b, c in zip(tab2.mean_diff, tab2.ci_lo, tab2.ci_hi)]
    tab2["rel. impr. (%)"] = tab2.rel_improvement_pct.map(lambda v: f"{v:.1f}")
    tab2["PINN better"] = tab2.ours_better_frac.map(lambda v: f"{v:.2f}")
    tab2["p (Holm)"] = tab2.p_holm.map(lambda v: f"{v:.2g}")
    tab2 = tab2.set_index(tab2.metric + ": " + tab2.competitor)[["Δ (competitor − PINN)", "rel. impr. (%)", "PINN better", "p (Holm)"]]
    tab2.index.name = "Comparison"
    latex_table(tab2, TAB / "table2_significance.tex",
                "Paired comparison against the fine-tuned PINN (profile-level Wilcoxon signed-rank, Holm-corrected).", "tab:sig")
    display(tab2)

uq = UQ.set_index("method")
tab3 = pd.DataFrame({c: uq[c].map(lambda v: f"{v:.2f}") for c in uq.columns
                     if c.endswith(("cov50", "cov90", "cov95")) or c.startswith("spearman")}); tab3.index.name = "Method"
latex_table(tab3, TAB / "table3_uncertainty.tex", "Empirical coverage of central predictive intervals and Spearman correlation between predicted σ and absolute error.", "tab:uq")
print("Tables written to", TAB)

## 11. Auto-generated results summary
The text below is generated from the measured numbers. Statements of superiority are made **only** where the
Holm-corrected test is significant.

In [ ]:
lines = []
best = S_OVERALL.sort_values("T_MAE").iloc[0]
o = S_OVERALL.set_index("method")
lines.append(f"Test set: {len(TEST_IDS)} unseen profiles, {M_MAIN.case_id.nunique()} gap cases "
             f"({', '.join(map(str, CFG['GAP_SIZES_KM']))} km; {', '.join(CFG['GAP_LOCATIONS'])}).")
if len(TEST_IDS) < 6:
    lines.append(f"NOTE: only {len(TEST_IDS)} test profiles — a Wilcoxon test cannot reach p < 0.05 with fewer than 6; "
                 "this run is a smoke test, not a result.")
lines.append(f"Lowest overall temperature MAE: {best.method} ({best.T_MAE:.2f} K, 95% CI {best.T_MAE_lo:.2f}–{best.T_MAE_hi:.2f}).")
if OURS in o.index:
    r = o.loc[OURS]
    lines.append(f"{OURS}: T MAE {r.T_MAE:.2f} K, P error {r.P_relMAE:.2f} %, n error {r.n_relMAE:.2f} %, "
                 f"{100 * r.T_within_2sig:.0f} % of levels within 2σ of the retrieval.")
for met, lab in [("T_MAE", "temperature MAE"), ("P_relMAE", "pressure error"), ("n_relMAE", "density error")]:
    s = SIG[SIG.metric == met]
    better = s[s.significant & (s.mean_diff > 0)].competitor.tolist()
    worse = s[s.significant & (s.mean_diff < 0)].competitor.tolist()
    ns = s[~s.significant].competitor.tolist()
    lines.append(f"[{lab}] PINN significantly better than: {better or 'none'}; significantly worse than: {worse or 'none'}; "
                 f"no significant difference from: {ns or 'none'}.")
def _verdict(s):
    return ("better" if s.mean_diff > 0 else "worse") if s.significant else "not significantly different"
if len(SIG_LOC):
    for loc in CFG["GAP_LOCATIONS"]:
        for comp, met, unit in [(NAMES["mlp_rft"], "T_MAE", "K"), (NAMES["hydro"], "P_relMAE", "%"),
                                (NAMES["eof"], "T_MAE", "K"), (NAMES["pinn_ft"], "T_MAE", "K")]:
            s = SIG_LOC[(SIG_LOC.group == loc) & (SIG_LOC.metric == met) & (SIG_LOC.competitor == comp)]
            if len(s):
                s = s.iloc[0]
                lines.append(f"[{loc} gaps] {OURS} vs {comp} ({met}): {_verdict(s)} "
                             f"(Δ = {s.mean_diff:.3f} {unit}, CI {s.ci_lo:.3f}–{s.ci_hi:.3f}, p_Holm = {s.p_holm:.2g}).")
lines.append(f"[selection] Hyper-parameters chosen on validation profiles: {HP}.")
if HP["PINN"] == dict(lam=CFG["LAMBDA_PHYS"], alpha=0.0, es=False):
    lines.append("[selection] NOTE: validation selected settings identical to the plain fine-tuned PINN, so the two "
                 "rows are the same model; regularisation did not help on validation profiles.")
u = UQ.set_index("method")
for m in [NAMES["gp"], NAMES["eof"], NAMES["pinn_rft"]]:
    if m in u.index and "T_ccov95" in u and np.isfinite(u.loc[m, "T_ccov95"]):
        lines.append(f"[uncertainty, conformal] {m}: calibrated 95 % interval covers {100 * u.loc[m, 'T_ccov95']:.0f} % of T "
                     f"and {100 * u.loc[m, 'P_ccov95']:.0f} % of ln P values on test profiles.")
    if m in u.index and "T_cov95" in u:
        cov = u.loc[m, "T_cov95"]
        verdict = "close to nominal" if abs(cov - 0.95) <= 0.05 else ("over-confident (intervals too narrow)" if cov < 0.90 else "under-confident (intervals too wide)")
        lines.append(f"[uncertainty] {m}: nominal 95 % interval covers {100 * cov:.0f} % of T values — {verdict}; "
                     f"Spearman(σ, |e|) = {u.loc[m, 'spearman_T']:.2f}.")
if OURS in o.index:
    lines.append(f"[physics] Hydrostatic residual RMS: {OURS} {o.loc[OURS, 'Hydro_RMS']:.3g}, data-only MLP (reg.) "
                 f"{o.loc[NAMES['mlp_rft'], 'Hydro_RMS']:.3g}, reference floor {REF_HYDRO_FLOOR:.3g}.")
summary_text = "\n".join("- " + l for l in lines)
(OUT / "results_summary.md").write_text(summary_text)
print(summary_text)
print(f"\nTotal runtime: {(time.time() - T_START) / 60:.1f} min. All outputs in {OUT.resolve()}")

## Limitations
1. **Scope of the data.** All profiles are pre-dawn (≈03–05 LTST) ingress occultations, $L_s$ ≈ 21–142°
   (northern spring–summer, low-dust season), latitudes ≈ 13°S–69°N. Conclusions apply to this regime.
2. **Reference is a retrieval, not ground truth.** It carries uncertainty (σ_T, σ_P) and was itself produced with
   hydrostatic balance and the ideal-gas law; physical-consistency results measure agreement with the retrieval physics.
3. **Physics constrains P given T, not T itself.** Inside a gap, temperature comes from the learned cross-profile prior
   and the observed data. Any temperature improvement over the data-only network should be interpreted with this in mind.
4. **Synthetic gaps.** Gaps are imposed on complete profiles; real data loss may have different structure.
5. **Trimmed top.** Levels with σ_T above the threshold were removed, so results do not cover the
   upper-boundary initialisation zone.
6. **Confounded campaigns.** Mars year, season and latitude band change together between observing campaigns, so
   the Mars-year hold-out (E4) cannot separate interannual, seasonal and latitudinal transfer.